# BKQuiz · Qwen2.5-7B-Instruct → lời giải và câu hỏi từ lý thuyết
Chọn **Runtime → Change runtime type → T4 GPU** (hoặc GPU CUDA ≥14 GiB).
Teacher được cố định revision và chạy NF4. Student vẫn là Qwen3-1.7B cùng adapter CPT27.

Mặc định chỉ chạy 3 jobs kiểm tra trước. Sau đó đổi `MAX_JOBS=None` để chạy hết gói pilot 63 jobs.
Chỉ chuyển `PACKET_KIND="full"` sau khi xem chất lượng pilot; toàn bộ kế hoạch có 1.994 jobs.
Mỗi job gồm bản nháp, hai lần giải độc lập đổi thứ tự lựa chọn và một lần tự kiểm tra.
Cùng một model tự kiểm tra không tương đương chuyên gia độc lập. Mọi kết quả vẫn phải duyệt trên web.
Bộ nhớ đệm và kết quả lưu trên Drive sau từng job; chạy lại để tiếp tục. Không cắt cụt nguồn/đáp án cho vừa token.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import hashlib, json, subprocess, sys, zipfile, os
BASE = Path('/content/drive/MyDrive/BKQuiz_CPT/SFT_PEDAGOGY_QWEN7B_20261025')
ASSETS = {'runtime_sha256': 'd13f151b65194cce1550afea11ab5979f048464cc18175103d660281b24e7469', 'pilot_sha256': '2eb39d213876446b1ef2bec879b7280dc08179a505f2887a0a1e99f5693348fe', 'full_sha256': 'caa9a1f6f96411c9b6b9a8e096bc8cf97c59561cb1019eeee77d1e6ead7832e3'}
PACKET_KIND = 'pilot'  # 'pilot' hoặc 'full'
MAX_JOBS = 3  # None để chạy hết gói đã chọn
RETRY_FAILED = False
assert BASE.is_dir(), f'Chưa có thư mục dữ liệu: {BASE}'
print('Drive:', BASE)

In [ ]:
# Kiểm tra tệp trước khi cài đặt hoặc thực thi mã từ gói đã đóng băng.
def checked_file(name, expected):
    path = BASE / name
    with path.open('rb') as stream:
        actual = hashlib.file_digest(stream, 'sha256').hexdigest()
    if actual != expected:
        raise ValueError(f'SHA256 không khớp: {name}; tải lại đúng gói, không sửa hash trong notebook')
    return path
runtime_zip = checked_file('teacher_runtime.zip', ASSETS['runtime_sha256'])
input_zip = checked_file('teacher_' + PACKET_KIND + '.zip', ASSETS[PACKET_KIND + '_sha256'])
WORK = Path('/content/bkquiz_pedagogy_runtime')
WORK.mkdir(exist_ok=True)
with zipfile.ZipFile(runtime_zip) as archive:
    expected = {'pedagogy_core.py','teacher_runtime.py','requirements.txt','runtime_manifest.json'}
    if set(archive.namelist()) != expected or any(i.file_size > 1024*1024 for i in archive.infolist()):
        raise ValueError('Gói runtime không hợp lệ')
    raw = {n: archive.read(n) for n in expected}
    manifest = json.loads(raw['runtime_manifest.json'])
    for name, wanted in manifest['files_sha256'].items():
        if hashlib.sha256(raw[name]).hexdigest() != wanted:
            raise ValueError('Runtime file bị thay đổi: ' + name)
    for name, body in raw.items():
        (WORK / name).write_bytes(body)
print('Verified runtime/input:', runtime_zip.name, input_zip.name)

In [ ]:
# Môi trường riêng tránh ghi đè thư viện của kernel Colab đang chạy.
VENV = Path('/content/bkquiz_pedagogy_venv')
PYTHON = VENV / 'bin/python'
if not PYTHON.exists():
    subprocess.run([sys.executable, '-m', 'venv', '--system-site-packages', str(VENV)], check=True)
subprocess.run([str(PYTHON), '-m', 'pip', 'install', '--quiet', 'torch==2.6.0',
                '--index-url', 'https://download.pytorch.org/whl/cu124'], check=True)
# Không dùng vision/audio trong teacher văn bản này. Bỏ các bản hệ thống
# khỏi kiểm tra optional dependency để tránh ABI mismatch với torch riêng.
guard = WORK / 'sitecustomize.py'
guard.write_text("import sys\nimport importlib.util\n_original = importlib.util.find_spec\ndef _find(name, *args, **kwargs):\n    if name.split('.')[0] in {'torchvision','torchaudio','librosa'}: return None\n    return _original(name, *args, **kwargs)\nimportlib.util.find_spec = _find\n", encoding='utf-8')
subprocess.run([str(PYTHON), '-m', 'pip', 'install', '--quiet', '-r', str(WORK/'requirements.txt')], check=True)
env = dict(os.environ, PYTHONPATH=str(WORK), TOKENIZERS_PARALLELISM='false')
subprocess.run([str(PYTHON), '-c', 'import torch, transformers, bitsandbytes; print(torch.__version__, transformers.__version__); assert torch.cuda.is_available(); print(torch.cuda.get_device_properties(0))'], env=env, check=True)

In [ ]:
OUTPUT = BASE / 'runs' / PACKET_KIND
command = [str(PYTHON), '-u', str(WORK/'teacher_runtime.py'), '--input', str(input_zip), '--out', str(OUTPUT)]
if MAX_JOBS is not None:
    command += ['--max-jobs', str(MAX_JOBS)]
if RETRY_FAILED:
    command += ['--retry-failed']
subprocess.run(command, env=env, cwd=str(WORK), check=True)

In [ ]:
from google.colab import files
results = sorted(OUTPUT.glob('pedagogy_results_*.zip'), key=lambda p:p.stat().st_mtime)
if not results:
    raise RuntimeError('Chưa có kết quả thật; kiểm tra GPU và log phía trên')
print('Kết quả được lưu trên Drive:', results[-1])
print('Tải ZIP này lên http://127.0.0.1:8765/sft/pedagogy/ để duyệt từng câu.')
files.download(str(results[-1]))

## Duyệt và xuất SFT
Tải `pedagogy_results_*.zip` lên màn hình duyệt trên máy Windows. Kiểm tra lý thuyết gốc,
đáp án và giải thích từng phương án, sau đó chấp nhận/sửa/loại.
Chỉ những bản mới nhất đã được người dùng chấp nhận mới có mặt trong bản xuất SFT.
Train ưu tiên ≥80% câu hỏi mới từ lý thuyết, tối đa 20% câu hỏi cũ được bổ sung lời giải.
Validation/test giữ theo nhóm trang và nhóm câu hỏi; các PDF này đã xuất hiện trong CPT.
Notebook này **không huấn luyện student**, không tự ghi quyết định duyệt.